# MEL city bikes ETL

This notebook is part of data engineering project concentrated on building **ETL pipeline** using open datasets from the **detaMEL/Lille Metropole Open Data**.

---

## Table of Contents
1. Introduction
2. Environment Setup
3. Data Cleaning
4. Next Steps

## 1. Introduction

This section of the project covers **raw data cleaning** , based on insights gathered during the **initial EDA phase** of the **MEL city bikes** datasets.

---

**Licence and Usage**

These datasets are licensed under a **Licence Ouverte Etalab 2.0**.
They were downloaded on **3 December 2025**.


**Reference**

- Ilévia - *Comptages vélos depuis 2013*, dataMEL, [link↗](https://data.lillemetropole.fr/catalogue/dataset/fb7cf2fb-9466-48eb-a141-3869a4c1860b)
- Ilévia -*Populations des communes et de la MEL*, dataMEL, [link↗](https://data.lillemetropole.fr/catalogue/dataset/1106c6dc-a5ec-4eb8-a4e0-9cc813a2aeda)
- Ilévia -*Communes et communes associées de la Métropole Européenne de Lille*, dataMEL, [link↗](https://data.lillemetropole.fr/catalogue/dataset/25e38054-3194-46ab-ab88-f2b144ef26ed)

---

**Objective**

The goal of this stage is to prepare the raw datasets for the subsequent steps of the **ETL pipeline** by applying data cleaning operations.

This preparation supports later phases such as schema design, database integration, and ETL development (Talend).

## 2. Environment Setup

In this section, the Python environment is initialized to enable working with the raw data. Firstly, **pandas** library is imported as the primary tool for loading, inspecting and manipulating structured data as part of the ETL workflow.

The following datasets are loaded into separate dataframes for for separate inspection:

- `velos_df` - spot counts of bicycles on the MEL cycle network using an electromagnetic detection system installed in the road surface at the stations,
- `populations_df` - municipal and metropolitan population for years 2006, 2015 ans 2020,
- `communes_df` - the municipalities of MEL.

This operation allows a clearer understanding of the structure, schema, column meanings, and potential data quality issues before designing the final data model.

In [1]:
import pandas as pd

In [2]:
velos_df = pd.read_csv("../data/raw/velos.csv")
populations_df = pd.read_csv("../data/raw/populations.csv")
communes_df = pd.read_csv("../data/raw/communes.csv")

## 3. Data Cleaning and Exporting

### Bikes detection

From `velos_df`, which serve as the **future fact table**, **two dimension tables** were created: 
    - `capteurs_df`, containing the primary key and geospatial attributes of electromagnetic sensors.
    - `calendrier_df`, containing time-related information.

The `velos_df` was limited to sensor and time identifiers and aggregated sensor readings.

Missing values in `code_insee` were filled with **59009** (the city of Villeneuve-d'Ascq) based on EDA conclusions from `velos_df` and `capteurs_df`.

Duplicated information across tables was removed (e.g. columns `nom` and `geom`).

In [3]:
velos_df.head(3)

,id,nom,adresse,ville,code_insee,annee,semaine,mjo,longitude,latitude,geom
0,VEL012,Lille - Av. Le Corbusier,Lille - Av. Le Corbusier,LILLE,59350.0,2013,36,152.0,3.073626,50.638620,POINT (3.073625583618273 50.63862003542886)
1,VEL014,Lille - Grand Boulevard,Lille - Grand Boulevard,LILLE,59350.0,2013,36,156.0,3.087474,50.656063,POINT (3.087473889889443 50.6560630327743)
2,VEL012,Lille - Av. Le Corbusier,Lille - Av. Le Corbusier,LILLE,59350.0,2013,37,1973.0,3.073626,50.638620,POINT (3.073625583618273 50.63862003542886)


In [4]:
capteurs_df = velos_df.copy()
capteurs_df.head()

,id,nom,adresse,ville,code_insee,annee,semaine,mjo,longitude,latitude,geom
0,VEL012,Lille - Av. Le Corbusier,Lille - Av. Le Corbusier,LILLE,59350.0,2013,36,152.0,3.073626,50.638620,POINT (3.073625583618273 50.63862003542886)
1,VEL014,Lille - Grand Boulevard,Lille - Grand Boulevard,LILLE,59350.0,2013,36,156.0,3.087474,50.656063,POINT (3.087473889889443 50.6560630327743)
2,VEL012,Lille - Av. Le Corbusier,Lille - Av. Le Corbusier,LILLE,59350.0,2013,37,1973.0,3.073626,50.638620,POINT (3.073625583618273 50.63862003542886)
3,VEL014,Lille - Grand Boulevard,Lille - Grand Boulevard,LILLE,59350.0,2013,37,661.0,3.087474,50.656063,POINT (3.087473889889443 50.6560630327743)
4,VEL012,Lille - Av. Le Corbusier,Lille - Av. Le Corbusier,LILLE,59350.0,2013,38,1280.0,3.073626,50.638620,POINT (3.073625583618273 50.63862003542886)


In [5]:
capteurs_df.drop(columns = ['nom', 'annee', 'semaine', 'mjo', 'geom'], inplace = True)
capteurs_df.head()

,id,adresse,ville,code_insee,longitude,latitude
0,VEL012,Lille - Av. Le Corbusier,LILLE,59350.0,3.073626,50.638620
1,VEL014,Lille - Grand Boulevard,LILLE,59350.0,3.087474,50.656063
2,VEL012,Lille - Av. Le Corbusier,LILLE,59350.0,3.073626,50.638620
3,VEL014,Lille - Grand Boulevard,LILLE,59350.0,3.087474,50.656063
4,VEL012,Lille - Av. Le Corbusier,LILLE,59350.0,3.073626,50.638620


In [6]:
capteurs_df.drop_duplicates(inplace = True)
capteurs_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 37 entries, 0 to 9885
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          37 non-null     object 
 1   adresse     37 non-null     object 
 2   ville       37 non-null     object 
 3   code_insee  35 non-null     float64
 4   longitude   37 non-null     float64
 5   latitude    37 non-null     float64
dtypes: float64(3), object(3)
memory usage: 2.0+ KB


In [7]:
years = range(2004, 2026)

calendar_rows = []

for year in years:
    num_weeks = pd.Timestamp(f'{year}-12-28').isocalendar()[1]
    for week in range(1, num_weeks + 1):
        calendar_rows.append({'annee' : year, 'semaine' : week})

calendrier_df = pd.DataFrame(calendar_rows)

calendrier_df['annee_semaine'] = calendrier_df['annee'].astype(str) + '_' + calendrier_df['semaine'].astype(str)
calendrier_df.head()

,annee,semaine,annee_semaine
0,2004,1,2004_1
1,2004,2,2004_2
2,2004,3,2004_3
3,2004,4,2004_4
4,2004,5,2004_5


In [8]:
velos_df['annee_semaine'] = velos_df['annee'].astype(str) + '_' + velos_df['semaine'].astype(str)
velos_df.drop(columns = ['nom', 'adresse', 'ville', 'annee', 'semaine', 'longitude', 'latitude', 'geom'], inplace = True)
velos_df.head()

,id,code_insee,mjo,annee_semaine
0,VEL012,59350.0,152.0,2013_36
1,VEL014,59350.0,156.0,2013_36
2,VEL012,59350.0,1973.0,2013_37
3,VEL014,59350.0,661.0,2013_37
4,VEL012,59350.0,1280.0,2013_38


In [9]:
capteurs_df['code_insee'].fillna(59009, inplace = True)
velos_df['code_insee'].fillna(59009, inplace = True)

In [10]:
velos_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11398 entries, 0 to 11397
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             11398 non-null  object 
 1   code_insee     11398 non-null  float64
 2   mjo            11398 non-null  float64
 3   annee_semaine  11398 non-null  object 
dtypes: float64(2), object(2)
memory usage: 356.3+ KB


In [11]:
capteurs_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 37 entries, 0 to 9885
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          37 non-null     object 
 1   adresse     37 non-null     object 
 2   ville       37 non-null     object 
 3   code_insee  37 non-null     float64
 4   longitude   37 non-null     float64
 5   latitude    37 non-null     float64
dtypes: float64(3), object(3)
memory usage: 2.0+ KB


### Population changes

It was decided to remove four columns from the `populations_df` dataset:

- `epci`, and `scot` due to lack of meaningful information for further analysis,
- `evo_pop_06_15`, `evo_pop_06_20`, and `evo_pop_15_20` because they are derived from the base population columns (`pop_2006`, `pop_2015`, and `pop_2020`),
- `surface_m2`, and `geom`, since this information is already contained in the `communes_df` dataset.

Additionally, the columns `pop_2006`, `pop_2015`, and `pop_2020` were combined into a single population column with an accompanying year column (long format).

In [12]:
populations_df.head(3)

,code_adm,code_insee,nom,epci,scot,pop_2006,pop_2015,pop_2020,evo_pop_06_15,evo_pop_06_20,evo_pop_15_20,surface_m2,geom
0,COM_F59005,59005,Allennes-les-Marais,EPCI_245900410,SCOT_1000,3407,3413,3532,6,125,119,5.561176e+06,MULTIPOLYGON (((2.9630895928204106 50.52802738...
1,COM_F59009,59009,Villeneuve-d'Ascq,EPCI_245900410,SCOT_1000,61151,61920,61258,769,107,-662,2.757522e+07,MULTIPOLYGON (((3.1793022723575555 50.60930052...
2,COM_F59011,59011,Annœullin,EPCI_245900410,SCOT_1000,9802,10407,10679,605,877,272,9.008851e+06,MULTIPOLYGON (((2.8954464720510598 50.53189944...


In [13]:
populations_df = populations_df.melt(
    id_vars=['code_adm', 'code_insee'],
    value_vars=['pop_2006', 'pop_2015', 'pop_2020'],
    var_name='annee',
    value_name='population'
)

populations_df['annee'] = populations_df['annee'].str.replace('pop_', '').astype(int)

populations_df.head()

,code_adm,code_insee,annee,population
0,COM_F59005,59005,2006,3407
1,COM_F59009,59009,2006,61151
2,COM_F59011,59011,2006,9802
3,COM_F59013,59013,2006,1195
4,COM_F59017,59017,2006,24836


In [14]:
populations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   code_adm    294 non-null    object
 1   code_insee  294 non-null    int64 
 2   annee       294 non-null    int32 
 3   population  294 non-null    int64 
dtypes: int32(1), int64(2), object(1)
memory usage: 8.2+ KB


### Municipalities details

Some of the columns wered dropped:

- `code_gdt`, and `libelle_gdt` as they were empty,
- `population`, `densite_pop_km2`, and `source_mil` as they concern the number of inhabitants which change in time and is already tracked in `populations_df`,
- `nom_etq2`, and `nom_etq3` because they duplicate the information in concerning the municipalities.

In case of missing data in the object-type columns containing administrative flag information **NA** was inputed.

In [15]:
communes_df.head(3)

,code_adm,code_insee,nature,nom,nom_etq2,nom_etq3,code_mel,libelle_mel,code_ut,libelle_ut,...,code_dt,libelle_dt,code_gdt,libelle_gdt,surface_ha,surface_km2,population,densite_pop_km2,source_mil,geom
0,COM_F59005,59005,Commune,Allennes-les-Marais,Allennes-les-Marais,Allennes-\n-les-Marais,MEL,Métropole Européenne de Lille,UTLS,UT Lille / Seclin,...,NaN,NaN,NaN,NaN,554,5.54,3460,624.55,IGN BD_TOPO 2020(2019) + INSEE 2020(2017),MULTIPOLYGON (((2.9630895928204106 50.52802738...
1,COM_F59009,59009,Commune,Villeneuve-d'Ascq,Villeneuve-d'Ascq,Villeneuve-\n-d'Ascq,MEL,Métropole Européenne de Lille,UTRV,UT Roubaix / Villeneuve d'Ascq,...,NaN,NaN,NaN,NaN,2748,27.48,63408,2307.42,IGN BD_TOPO 2020(2019) + INSEE 2020(2017),MULTIPOLYGON (((3.1793022723575555 50.60930052...
2,COM_F59011,59011,Commune,Annœullin,Annœullin,Annœullin,MEL,Métropole Européenne de Lille,UTLS,UT Lille / Seclin,...,NaN,NaN,NaN,NaN,898,8.98,10428,1161.25,IGN BD_TOPO 2020(2019) + INSEE 2020(2017),MULTIPOLYGON (((2.8954464720510598 50.53189944...


In [16]:
communes_df.drop(columns= ['nom_etq2', 'nom_etq3', 'population', 'densite_pop_km2', 'source_mil', 'code_gdt', 'libelle_gdt'], inplace = True)

In [17]:
cols = communes_df.select_dtypes(include=['object']).columns
communes_df[cols] = communes_df[cols].fillna('NA')

In [18]:
communes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 23 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   code_adm                        97 non-null     object 
 1   code_insee                      97 non-null     int64  
 2   nature                          97 non-null     object 
 3   nom                             97 non-null     object 
 4   code_mel                        97 non-null     object 
 5   libelle_mel                     97 non-null     object 
 6   code_ut                         97 non-null     object 
 7   libelle_ut                      97 non-null     object 
 8   code_territoire_gouvernance     97 non-null     object 
 9   libelle_territoire_gouvernance  97 non-null     object 
 10  code_deveco                     97 non-null     object 
 11  libelle_deveco                  97 non-null     object 
 12  code_enm                        97 non

All dataframes have been cleaned and saved for review. Final operations and export will be executed in Talend.

In [19]:
velos_df.to_csv("../data/processed/velos_cleaned.csv", index=False)
capteurs_df.to_csv("../data/processed/capteurs_cleaned.csv", index=False)
calendrier_df.to_csv("../data/processed/calendrier_cleaned.csv", index=False)
populations_df.to_csv("../data/processed/populations_cleaned.csv", index=False)
communes_df.to_csv("../data/processed/communes_cleaned.csv", index=False)

## 4. Preliminary Observations

- From the `velos_df` **two dimension tables** are separated: `dim_calendrier` and `dim_capteur`. Third **dimension table**  is created directly from `communes_df` as `dim_commune`.

- Two fact tables were identified: `fact_velo` from `velos_df` and `fact_population` from `populations_df`. Both are related to the `dim_commune` dimension table using its primary key `code_insee`. They were limited only to the measures and identifiers -  the foreign keys.

## 5. Next Steps

Based on the initial data exploration and profiling, the following steps will be performed in the next phase of the ETL project:

1. **Draft Star Schema**
   - Create a preliminary **star schema** diagram showing:
     - Fact tables (`fact_velo`, `fact_population`)
     - Connected dimension tables (`dim_commune`, `dim_calendrier`, `dim_capteur`)
   - Use this schema as a blueprint for the **PostgreSQL database** structure.

2. **Database Implementation**
   - Create empty PostgreSQL tables according to the draft schema.
   - Define primary keys, constraints, and column types.
  
3. **ETL Pipeline (Talend)**
   Implement data extraction, transformation, and loading from CSVs:

    - Create a calendar and a sensor tables.
    - Handle missing values in key columns using appropriate strategies (imputation).
    - Map and confirm key relationships:
        - `fact_velo.code_insee` → `dim_commune.code_insee`
        - `fact_population.code_insee` → `dim_commune.code_insee`
    - Finalize and enforce primary keys:
        - `dim_commune`: `code_isnee`
        - `dim_calendrier`: `calendrier_id`
        - `dim_capteur: capteur_id`
   - Remove redundant and constant fields (e.g., `epci`, `scot`, `adresse`)
     

5. **Future Considerations**
   - Integrate `fact_population` with  `dim_calendrier` eshaping the population dataset into a long format.
   - Schedule automated ETL execution (e.g. cron-based orchestration).
   - Connect the PostgreSQL database wit Power BI to build an interactive dashboard.
   - Enrich the data model with contextual datasets (e.g., schools, parks, hospitals) to improve analytical depth.